# GridPulse BR — Silver Transformation

## Objective

Transform the ANEEL continuity indicators Bronze dataset into a trusted
Silver layer suitable for downstream analytical consumption.

The Silver layer improves semantic consistency and data quality while
preserving the business meaning of the original source.

## Silver responsibilities

- Standardize business identifiers and data types
- Remove exact duplicate business observations
- Validate the business grain
- Validate indicator and reporting-period domains
- Preserve source lineage
- Persist a trusted Delta table for downstream analytical use

## Out of scope

The Silver layer does not:

- apply regulatory limits
- calculate DEC/FEC performance metrics
- rank distributors
- create analytical KPIs
- aggregate business results

These responsibilities belong to the Gold layer.

In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "workspace.gridpulse.bronze_aneel_continuity_indicators"
TARGET_TABLE = "workspace.gridpulse.silver_aneel_continuity_indicators"

BUSINESS_KEY = [
    "IdeConjUndConsumidoras",
    "SigIndicador",
    "AnoIndice",
    "NumPeriodoIndice"
]

print(f"Source table : {SOURCE_TABLE}")
print(f"Target table : {TARGET_TABLE}")
print(f"Business key : {BUSINESS_KEY}")

## 1. Read Bronze data

Read the persisted Bronze dataset that will serve as the source for the
Silver transformation.

In [0]:
df_bronze = spark.table(SOURCE_TABLE)

bronze_row_count = df_bronze.count()

print(f"Bronze rows    : {bronze_row_count:,}")
print(f"Bronze columns : {len(df_bronze.columns)}")

df_bronze.printSchema()

## 2. Validate duplicate business observations

Confirm that the duplicate behavior identified during source profiling is
still present in the Bronze layer before any records are removed.

The known duplicate pattern consists of exact duplicate business
observations.

In [0]:
duplicate_profile = (
    df_bronze
    .groupBy(*BUSINESS_KEY)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("VlrIndiceEnviado").alias("distinct_values"),
        F.countDistinct("DatGeracaoConjuntoDados").alias("generation_dates")
    )
    .filter(F.col("row_count") > 1)
)

duplicate_group_count = duplicate_profile.count()

print(f"Duplicate business-key groups : {duplicate_group_count:,}")

display(
    duplicate_profile
    .groupBy(
        "row_count",
        "distinct_values",
        "generation_dates"
    )
    .count()
    .orderBy(
        "row_count",
        "distinct_values",
        "generation_dates"
    )
)

## 3. Standardize and deduplicate business observations

The Silver layer removes exact duplicate business observations and
standardizes selected business attributes.

The business grain is defined by:

- consumer-unit set
- indicator
- year
- reporting period

CNPJ is normalized as a 14-character string because it is a business
identifier rather than a numeric measure.

In [0]:
df_silver = (
    df_bronze
    .dropDuplicates(BUSINESS_KEY)
    .withColumn(
        "NumCNPJ",
        F.lpad(
            F.col("NumCNPJ").cast("string"),
            14,
            "0"
        )
    )
)

silver_row_count = df_silver.count()

print(f"Bronze rows : {bronze_row_count:,}")
print(f"Silver rows : {silver_row_count:,}")
print(f"Rows removed as exact duplicates : {bronze_row_count - silver_row_count:,}")

## 4. Validate CNPJ normalization

In [0]:
cnpj_validation = (
    df_silver
    .agg(
        F.min(F.length("NumCNPJ")).alias("min_length"),
        F.max(F.length("NumCNPJ")).alias("max_length"),
        F.sum(
            F.when(
                ~F.col("NumCNPJ").rlike("^[0-9]{14}$"),
                1
            ).otherwise(0)
        ).alias("invalid_cnpj_count")
    )
)

display(cnpj_validation)

## 5. Validate Silver business grain

In [0]:
silver_duplicate_keys = (
    df_silver
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

remaining_duplicate_groups = silver_duplicate_keys.count()

print(f"Remaining duplicate business-key groups : {remaining_duplicate_groups:,}")

assert remaining_duplicate_groups == 0, (
    f"Silver business grain validation failed: "
    f"{remaining_duplicate_groups:,} duplicate groups remain."
)

print("Silver business grain validation passed.")

## 6. Validate business domains

Validate core domain expectations before publishing the Silver dataset.

These checks ensure that the standardized dataset remains within the known
business and temporal boundaries of the source.

In [0]:
domain_profile = (
    df_silver
    .agg(
        F.min("AnoIndice").alias("min_year"),
        F.max("AnoIndice").alias("max_year"),
        F.min("NumPeriodoIndice").alias("min_period"),
        F.max("NumPeriodoIndice").alias("max_period"),
        F.min("VlrIndiceEnviado").alias("min_value"),
        F.max("VlrIndiceEnviado").alias("max_value"),
        F.countDistinct("SigIndicador").alias("indicator_count")
    )
)

display(domain_profile)

In [0]:
invalid_period_count = (
    df_silver
    .filter(
        (F.col("NumPeriodoIndice") < 1) |
        (F.col("NumPeriodoIndice") > 12)
    )
    .count()
)

invalid_year_count = (
    df_silver
    .filter(
        (F.col("AnoIndice") < 2020) |
        (F.col("AnoIndice") > 2029)
    )
    .count()
)

negative_value_count = (
    df_silver
    .filter(F.col("VlrIndiceEnviado") < 0)
    .count()
)

print(f"Invalid periods : {invalid_period_count:,}")
print(f"Invalid years   : {invalid_year_count:,}")
print(f"Negative values : {negative_value_count:,}")

In [0]:
assert invalid_period_count == 0, (
    f"Invalid reporting periods found: {invalid_period_count:,}"
)

assert invalid_year_count == 0, (
    f"Unexpected indicator years found: {invalid_year_count:,}"
)

print("Silver domain validation passed.")

## 7. Persist Silver Delta table

Persist the standardized and deduplicated dataset as a managed Delta table.

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Silver table created successfully: {TARGET_TABLE}")

## 8. Bronze-to-Silver reconciliation

Validate that the Silver row count equals the Bronze row count minus the
known exact duplicate records removed during transformation.

In [0]:
df_silver_persisted = spark.table(TARGET_TABLE)

persisted_silver_count = df_silver_persisted.count()
expected_silver_count = bronze_row_count - 260

print(f"Bronze rows          : {bronze_row_count:,}")
print(f"Expected Silver rows : {expected_silver_count:,}")
print(f"Actual Silver rows   : {persisted_silver_count:,}")

assert persisted_silver_count == expected_silver_count, (
    f"Silver reconciliation failed: expected "
    f"{expected_silver_count:,}, got {persisted_silver_count:,}"
)

print("Bronze-to-Silver reconciliation passed.")

In [0]:
print("=" * 60)
print("GRIDPULSE BR — SILVER TRANSFORMATION SUMMARY")
print("=" * 60)

print(f"Bronze rows                 : {bronze_row_count:,}")
print(f"Silver rows                 : {persisted_silver_count:,}")
print(f"Exact duplicates removed    : {bronze_row_count - persisted_silver_count:,}")
print(f"Remaining duplicate groups  : {remaining_duplicate_groups:,}")
print(f"Invalid periods             : {invalid_period_count:,}")
print(f"Invalid years               : {invalid_year_count:,}")
print(f"Negative indicator values   : {negative_value_count:,}")
print(f"Target table                : {TARGET_TABLE}")

print("=" * 60)
print("STATUS: SUCCESS")
print("=" * 60)